<a href="https://colab.research.google.com/github/anshajk/python-frameworks/blob/main/notebooks/Prompt_Engineering_for_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Prompt Engineering for Text Data Analysis: Demonstration**

## **Learning Objective**

This demo aims to teach how to effectively leverage Large Language Models (LLMs) for practical text analysis tasks through prompt engineering techniques. Students will learn how to design robust prompts that produce consistent, structured outputs for real-world applications.

## **Demo Overview**

This demonstration showcases two practical applications of LLMs for text analysis:

- Movie Review Sentiment Analysis
- Customer Support Ticket Classification


## **Key Prompt Engineering Techniques Demonstrated**

1. **System Role Definition**: Establishing clear roles for the model ("sentiment analysis expert", "customer support ticket classifier")

2. **Output Constraints**: Limiting responses to specific formats for consistent parsing

3. **Few-Shot Learning**: Providing examples that demonstrate the expected input-output relationship

4. **Chain-of-Thought Reasoning**: Breaking complex tasks into logical steps to improve accuracy

5. **Temperature Control**: Adjusting the randomness parameter to balance creativity and consistency

6. **Prompt Structure**: Organizing prompts with clear instructions, examples, and formatting guidelines

## **Practical Applications**

These techniques can be applied to numerous real-world scenarios:

- Content moderation and classification
- Customer feedback analysis
- Support ticket routing and prioritization
- Document categorization
- Automated response generation


In [ ]:
# Importing the OpenAI library to interact with OpenAI's API services.
from openai import OpenAI

In [ ]:
import os  # Importing the os module to interact with environment variables
import getpass  # Importing getpass to securely input sensitive information

# Prompting the user to securely enter their OpenAI API key without displaying it on the screen
OPENAI_API_KEY = getpass.getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key: ··········


In [ ]:
# Creating an instance of the OpenAI client using the provided API key.
client = OpenAI(api_key=OPENAI_API_KEY)

### **Part 1: Movie Review Sentiment Analysis**

In this section, we explore how to:
- Process a dataset of movie reviews to extract sentiment information
- Design effective prompts that produce consistent sentiment classifications
- Implement batch processing to handle larger datasets efficiently
- Use GPT-4 Mini to classify reviews as positive, negative, or neutral

The code demonstrates a complete workflow from data ingestion to sentiment classification, with practical considerations like API rate limiting and error handling.

In [ ]:
import pandas as pd
import openai
from time import sleep

# Create a sample DataFrame with movie reviews
reviews = [
    "This movie was absolutely fantastic! The acting was superb and the plot kept me engaged throughout. Definitely a must-watch!",
    "I was really disappointed with this film. The characters were poorly developed and the storyline made no sense. Complete waste of time.",
    "It was an okay movie. Nothing special but not terrible either. The visuals were nice but the story was predictable.",
    "One of the best films I've seen this year! The director's vision was perfectly executed and the soundtrack was phenomenal.",
    "Terrible acting, awful script, and the special effects looked cheap. I can't believe I paid money to see this.",
    "The movie had its moments, but overall it was just average. Some scenes were good while others dragged on too long.",
    "A masterpiece of modern cinema! Every aspect from cinematography to dialogue was perfectly crafted.",
    "I fell asleep halfway through. The pacing was too slow and the characters were forgettable.",
    "Not bad, not great. It's the kind of movie you watch once and then forget about.",
    "Absolutely loved it! The performances were Oscar-worthy and the story was both heartbreaking and uplifting."
]

# Create the DataFrame
df = pd.DataFrame({'review': reviews})

# Display the original DataFrame
print("Original DataFrame:")
df


Original DataFrame:


,review
0,This movie was absolutely fantastic! The actin...
1,I was really disappointed with this film. The ...
2,It was an okay movie. Nothing special but not ...
3,One of the best films I've seen this year! The...
4,"Terrible acting, awful script, and the special..."
5,"The movie had its moments, but overall it was ..."
6,A masterpiece of modern cinema! Every aspect f...
7,I fell asleep halfway through. The pacing was ...
8,"Not bad, not great. It's the kind of movie you..."
9,Absolutely loved it! The performances were Osc...


In [ ]:
example_df = pd.DataFrame({'review': reviews})
example_df['sentiment'] = ''
example_df

,review,sentiment
0,This movie was absolutely fantastic! The actin...,
1,I was really disappointed with this film. The ...,
2,It was an okay movie. Nothing special but not ...,
3,One of the best films I've seen this year! The...,
4,"Terrible acting, awful script, and the special...",
5,"The movie had its moments, but overall it was ...",
6,A masterpiece of modern cinema! Every aspect f...,
7,I fell asleep halfway through. The pacing was ...,
8,"Not bad, not great. It's the kind of movie you...",
9,Absolutely loved it! The performances were Osc...,


In [ ]:
def get_sentiment(review):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",  # Using gpt-4-mini as requested
            messages=[
                {"role": "system", "content": "You are a sentiment analysis expert. Classify the movie review sentiment as POSITIVE, NEGATIVE, or NEUTRAL. Respond with only one word: POSITIVE, NEGATIVE, or NEUTRAL."},
                {"role": "user", "content": review}
            ],
            temperature=0.3,  # Lower temperature for more consistent results
            max_tokens=10     # We only need one word
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error processing review: {e}")
        return "ERROR"

# Function to process the DataFrame in batches
def analyze_sentiments(df, batch_size=5):  # Smaller batch size for the example
    # Create a copy of the DataFrame
    df_with_sentiment = df.copy()

    # Initialize the sentiment column
    df_with_sentiment['sentiment'] = ''

    # Process reviews in batches
    for i in range(0, len(df), batch_size):
        batch = df_with_sentiment.iloc[i:i+batch_size]

        for idx, row in batch.iterrows():
            # Get sentiment for each review
            print(f"Analyzing review {idx+1}...")
            sentiment = get_sentiment(row['review'])
            df_with_sentiment.at[idx, 'sentiment'] = sentiment

            # Add a small delay to respect API rate limits
            sleep(0.5)

        print(f"Processed {min(i+batch_size, len(df))} out of {len(df)} reviews")

    return df_with_sentiment

# Run the sentiment analysis
# Uncomment the next line to actually run the analysis (requires valid API key)
df_with_sentiment = analyze_sentiments(df)

Analyzing review 1...
Analyzing review 2...
Analyzing review 3...
Analyzing review 4...
Analyzing review 5...
Processed 5 out of 10 reviews
Analyzing review 6...
Analyzing review 7...
Analyzing review 8...
Analyzing review 9...
Analyzing review 10...
Processed 10 out of 10 reviews


In [ ]:
df_with_sentiment

,review,sentiment
0,This movie was absolutely fantastic! The actin...,POSITIVE
1,I was really disappointed with this film. The ...,NEGATIVE
2,It was an okay movie. Nothing special but not ...,NEUTRAL
3,One of the best films I've seen this year! The...,POSITIVE
4,"Terrible acting, awful script, and the special...",NEGATIVE
5,"The movie had its moments, but overall it was ...",NEUTRAL
6,A masterpiece of modern cinema! Every aspect f...,POSITIVE
7,I fell asleep halfway through. The pacing was ...,NEGATIVE
8,"Not bad, not great. It's the kind of movie you...",NEUTRAL
9,Absolutely loved it! The performances were Osc...,POSITIVE


### **Part 2: Customer Support Ticket Classification**

This more advanced example demonstrates:
- Multi-dimensional classification (priority level and department)
- Implementation of chain-of-thought reasoning to improve classification accuracy
- Few-shot learning with carefully selected examples
- Structured output formatting for easier downstream processing

This example shows how to guide the model through a step-by-step reasoning process to make more nuanced classifications.

In [ ]:
import pandas as pd
import openai
from time import sleep

# Create a sample DataFrame with customer support tickets
tickets = [
    "I can't log into my account. I've tried resetting my password three times but I'm not receiving any email.",
    "Your product is amazing! Just wanted to let you know how much I love using it every day.",
    "The checkout process charged my credit card twice for the same order #45692. I need a refund immediately.",
    "How do I export my data to CSV format? I couldn't find this option in the settings menu.",
    "The mobile app keeps crashing whenever I try to upload a photo. This started happening after the latest update.",
    "I'd like to upgrade my subscription from basic to premium. What are the steps to do this?",
    "There seems to be a security vulnerability in your payment system. I'm a security researcher and found that...",
    "The loading time for the dashboard has been extremely slow for the past two days. Pages take 30+ seconds to load.",
    "I requested a refund 2 weeks ago (ticket #34267) but haven't received the money back in my account yet.",
    "Is your service compatible with Mac OS Sonoma? The documentation only mentions older versions."
]

# Create the DataFrame
df = pd.DataFrame({'ticket_text': tickets})

# Display the original DataFrame
print("Original Support Tickets:")
df

Original Support Tickets:


,ticket_text
0,I can't log into my account. I've tried resett...
1,Your product is amazing! Just wanted to let yo...
2,The checkout process charged my credit card tw...
3,How do I export my data to CSV format? I could...
4,The mobile app keeps crashing whenever I try t...
5,I'd like to upgrade my subscription from basic...
6,There seems to be a security vulnerability in ...
7,The loading time for the dashboard has been ex...
8,I requested a refund 2 weeks ago (ticket #3426...
9,Is your service compatible with Mac OS Sonoma?...


In [ ]:
import pandas as pd
import openai
from time import sleep

# Create a sample DataFrame with customer support tickets
tickets = [
    "I can't log into my account. I've tried resetting my password three times but I'm not receiving any email.",
    "Your product is amazing! Just wanted to let you know how much I love using it every day.",
    "The checkout process charged my credit card twice for the same order #45692. I need a refund immediately.",
    "How do I export my data to CSV format? I couldn't find this option in the settings menu.",
    "The mobile app keeps crashing whenever I try to upload a photo. This started happening after the latest update.",
    "I'd like to upgrade my subscription from basic to premium. What are the steps to do this?",
    "There seems to be a security vulnerability in your payment system. I'm a security researcher and found that...",
    "The loading time for the dashboard has been extremely slow for the past two days. Pages take 30+ seconds to load.",
    "I requested a refund 2 weeks ago (ticket #34267) but haven't received the money back in my account yet.",
    "Is your service compatible with Mac OS Sonoma? The documentation only mentions older versions."
]

# Create the DataFrame
df = pd.DataFrame({'ticket_text': tickets})

# Display the original DataFrame
print("Original Support Tickets:")
print(df)
print("\n")

def classify_ticket(ticket_text):
    try:
        # This prompt uses both few-shot examples and chain-of-thought reasoning
        prompt = """
You are an expert customer support ticket classifier. Analyze each ticket and classify it by priority (Low, Medium, High, Critical) and department (Account, Billing, Technical, Security, General).

Follow this step-by-step process:
1. Identify the main issue in the ticket
2. Determine if it affects core functionality, security, or financial aspects
3. Assess urgency based on customer language and impact
4. Assign appropriate priority level
5. Determine which department should handle this issue

Here are some examples with my reasoning:

Ticket: "I forgot my password and need to reset it."
Thinking: This is about account access but doesn't indicate repeated attempts or urgency. It's a common issue with a standard solution.
Priority: Low
Department: Account

Ticket: "Your website has been down for 3 hours and I can't place my order."
Thinking: This affects the core functionality (ordering) and is preventing sales. Multiple users are likely affected.
Priority: High
Department: Technical

Ticket: "I was charged twice for my subscription this month."
Thinking: This is a billing error affecting the customer financially. Needs prompt attention but isn't a system-wide emergency.
Priority: Medium
Department: Billing

Ticket: "I found a bug that lets me access other users' private information."
Thinking: This is a serious security vulnerability that could affect user privacy and company reputation.
Priority: Critical
Department: Security

Now classify this ticket:
{ticket}

Thinking step by step:
"""

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are an AI assistant that classifies customer support tickets."},
                {"role": "user", "content": prompt.format(ticket=ticket_text)}
            ],
            temperature=0.2,  # Lower temperature for more consistent results
            max_tokens=250    # Allow enough tokens for reasoning
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error processing ticket: {e}")
        return "ERROR"

# Function to process the DataFrame in batches
def analyze_tickets(df, batch_size=5):
    # Create a copy of the DataFrame
    df_with_classification = df.copy()

    # Initialize the classification columns
    df_with_classification['analysis'] = ''
    df_with_classification['priority'] = ''
    df_with_classification['department'] = ''

    # Process tickets in batches
    for i in range(0, len(df), batch_size):
        batch = df_with_classification.iloc[i:i+batch_size]

        for idx, row in batch.iterrows():
            # Get classification for each ticket
            print(f"Analyzing ticket {idx+1}...")
            analysis = classify_ticket(row['ticket_text'])
            df_with_classification.at[idx, 'analysis'] = analysis

            # Extract priority and department from the analysis
            # This is a simple extraction - could be improved with regex
            try:
                lines = analysis.split('\n')
                for line in lines:
                    if line.startswith('Priority:'):
                        df_with_classification.at[idx, 'priority'] = line.replace('Priority:', '').strip()
                    if line.startswith('Department:'):
                        df_with_classification.at[idx, 'department'] = line.replace('Department:', '').strip()
            except:
                # If extraction fails, leave as is
                pass

            # Add a small delay to respect API rate limits
            sleep(0.5)

        print(f"Processed {min(i+batch_size, len(df))} out of {len(df)} tickets")

    return df_with_classification

Original Support Tickets:
                                         ticket_text
0  I can't log into my account. I've tried resett...
1  Your product is amazing! Just wanted to let yo...
2  The checkout process charged my credit card tw...
3  How do I export my data to CSV format? I could...
4  The mobile app keeps crashing whenever I try t...
5  I'd like to upgrade my subscription from basic...
6  There seems to be a security vulnerability in ...
7  The loading time for the dashboard has been ex...
8  I requested a refund 2 weeks ago (ticket #3426...
9  Is your service compatible with Mac OS Sonoma?...




In [ ]:
tickets_classified = analyze_tickets(df)

Analyzing ticket 1...
Analyzing ticket 2...
Analyzing ticket 3...
Analyzing ticket 4...
Analyzing ticket 5...
Processed 5 out of 10 tickets
Analyzing ticket 6...
Analyzing ticket 7...
Analyzing ticket 8...
Analyzing ticket 9...
Analyzing ticket 10...
Processed 10 out of 10 tickets


In [ ]:
tickets_classified

,ticket_text,analysis,priority,department
0,I can't log into my account. I've tried resett...,1. Identify the main issue in the ticket: The ...,High,Account
1,Your product is amazing! Just wanted to let yo...,1. Identify the main issue in the ticket: The ...,Low,General
2,The checkout process charged my credit card tw...,1. Identify the main issue in the ticket: The ...,High,Billing
3,How do I export my data to CSV format? I could...,1. Identify the main issue in the ticket: The ...,Low,General
4,The mobile app keeps crashing whenever I try t...,1. Identify the main issue in the ticket: The ...,High,Technical
5,I'd like to upgrade my subscription from basic...,1. Identify the main issue in the ticket: The ...,Low,Account
6,There seems to be a security vulnerability in ...,1. Identify the main issue in the ticket: The ...,Critical,Security
7,The loading time for the dashboard has been ex...,1. Identify the main issue in the ticket: The ...,High,Technical
8,I requested a refund 2 weeks ago (ticket #3426...,1. Identify the main issue in the ticket: The ...,Medium,Billing
9,Is your service compatible with Mac OS Sonoma?...,1. Identify the main issue in the ticket: The ...,Low,Technical


In [ ]:
tickets_classified['ticket_text'].tolist()[6]

"There seems to be a security vulnerability in your payment system. I'm a security researcher and found that..."

In [ ]:
tickets_classified['analysis'].tolist()[6]

'1. Identify the main issue in the ticket: The ticket reports a potential security vulnerability in the payment system.\n\n2. Determine if it affects core functionality, security, or financial aspects: This issue directly pertains to security, as it involves the payment system, which is critical for financial transactions.\n\n3. Assess urgency based on customer language and impact: The language used indicates that the customer is a security researcher, which suggests that the issue may be serious and requires immediate attention to prevent potential exploitation.\n\n4. Assign appropriate priority level: Given the nature of the issue (security vulnerability) and the potential impact on user data and financial transactions, this should be classified as Critical.\n\n5. Determine which department should handle this issue: The appropriate department to handle a security vulnerability is the Security department.\n\nFinal classification:\nPriority: Critical\nDepartment: Security'

## **Key Takeaways**

1. Effective prompt design is crucial for getting consistent, useful outputs from LLMs

2. Complex classification tasks benefit from structured reasoning approaches

3. Few-shot examples significantly improve model performance without fine-tuning

4. Practical implementations must consider API limitations, error handling, and efficient processing

5. The same core techniques can be adapted to a wide variety of text analysis tasks

---
#  -----------------------------------------------------  **THANK YOU** ------------------------------------------------------------




---